# Week 65, Day 4: Caching

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w065d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from functools import lru_cache

calls = []

@lru_cache(maxsize=2)
def price(item):
    calls.append(item)
    return len(item) * 10

for item in ["tea", "jam", "tea", "milk", "jam", "tea"]:
    price(item)
print(calls)
print(price.cache_info().hits, price.cache_info().misses)

In [ ]:
from functools import cache

@cache
def tags_for(user):
    return ["new"]                 # a mutable result: shared by every caller

first = tags_for("ada")
first.append("admin")              # one caller changes "its" list...
print(tags_for("ada"))             # ...and the cached result is changed for everyone

@cache
def tags_for_safe(user):
    return ("new",)                # immutable: safe to share
print(tags_for_safe("ada") + ("admin",), tags_for_safe("ada"))

In [ ]:
import functools

def memoize(func):
    store, stats = {}, {"hits": 0, "misses": 0}

    @functools.wraps(func)
    def wrapper(*args):
        if args in store:
            stats["hits"] += 1
            return store[args]
        stats["misses"] += 1
        result = store[args] = func(*args)
        return result

    wrapper.stats = lambda: dict(stats, size=len(store))
    wrapper.cache_clear = lambda: (store.clear(), stats.update(hits=0, misses=0))
    return wrapper

@memoize
def ways(n):
    "Ways to climb n stairs taking 1 or 2 steps at a time."
    return 1 if n < 2 else ways(n - 1) + ways(n - 2)

print(ways(60), ways.stats(), ways.__doc__[:4])

In [ ]:
from collections import OrderedDict

class LRU:
    def __init__(self, capacity):
        self.capacity, self.data, self.evicted = capacity, OrderedDict(), []
    def get(self, key, default=None):
        if key not in self.data:
            return default
        self.data.move_to_end(key)                 # now the most recent
        return self.data[key]
    def put(self, key, value):
        self.data[key] = value
        self.data.move_to_end(key)
        if len(self.data) > self.capacity:
            self.evicted.append(self.data.popitem(last=False)[0])

c = LRU(2)
c.put("a", 1); c.put("b", 2); c.get("a"); c.put("c", 3)
print(list(c.data), c.evicted)

now = [0]
store = {}
def ttl_get(key, compute, ttl=10):
    if key in store and store[key][0] > now[0]:
        return store[key][1]
    value = compute()
    store[key] = (now[0] + ttl, value)
    return value

print(ttl_get("rate", lambda: 1.17), ttl_get("rate", lambda: 9.99))
now[0] = 11
print(ttl_get("rate", lambda: 1.18))

In [ ]:
import functools, hashlib, json

def freeze(value):
    if isinstance(value, (list, tuple)):
        return tuple(freeze(v) for v in value)
    if isinstance(value, dict):
        return tuple(sorted((k, freeze(v)) for k, v in value.items()))
    if isinstance(value, set):
        return frozenset(value)
    return value

def make_key(args, kwargs):
    return (freeze(args), freeze(kwargs))

print(make_key(([1, 2], {"b": 1, "a": 2}), {}) == make_key(((1, 2), {"a": 2, "b": 1}), {}))
try:
    functools.lru_cache()(len)([1, 2])
except TypeError as e:
    print("TypeError:", e)

text_key = json.dumps({"query": "tea", "page": 2}, sort_keys=True)
print(text_key, hashlib.sha256(text_key.encode()).hexdigest()[:12])

In [ ]:
import functools, gc

class Report:
    made = 0
    def __init__(self, rows):
        self.rows = rows
        Report.made += 1
    @functools.lru_cache(maxsize=None)
    def total(self):                    # self becomes part of a global cache key
        return sum(self.rows)
    @functools.cached_property
    def average(self):                  # stored on the instance, freed with it
        return sum(self.rows) / len(self.rows)

for i in range(3):
    r = Report([i, i + 1])
    r.total(), r.average
del r
gc.collect()
alive = sum(isinstance(o, Report) for o in gc.get_objects())
print(alive, Report.total.cache_info().currsize)
Report.total.cache_clear()
gc.collect()
print(sum(isinstance(o, Report) for o in gc.get_objects()))

> **Key idea.** Cache only results that depend just on their arguments, have no needed side effects, and won't be mutated by callers (return immutable values or copies); caching pays when calls are expensive and repeat often, so check the hit rate. Give caches statistics, cache_clear and functools.wraps; use functools.cache or lru_cache for hashable arguments. Bound caches by size (LRU: OrderedDict with move_to_end and popitem(last=False)) or by time (TTL, with an injectable clock). Keys must identify the question exactly: freeze unhashable arguments, sort keywords, include versions, and use strings for disk caches. Invalidate by not caching, expiring, clearing, versioning or tracking dependencies; avoid lru_cache on methods, which keeps instances alive.

### Exercise: A memoiser with statistics

Write a decorator `memoize(func)` that caches results by positional arguments **and** keyword arguments (with keyword order not mattering), and gives the wrapper `info()`, returning a dict with `hits`, `misses` and `size`, and `clear()`, which empties the cache and resets the counts. Keep the function's name and docstring.

In [ ]:
import functools

def memoize(func):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_memo():\n    \"memoising\"\n    calls = []\n    @memoize\n    def area(w, h=1, *, unit=\"cm\"):\n        \"Area of a rectangle.\"\n        calls.append((w, h, unit))\n        return f\"{w * h}{unit}2\"\n    assert area(2, 3) == \"6cm2\" and area(2, 3) == \"6cm2\" and area(2, h=3, unit=\"cm\") == \"6cm2\"\n    assert area(2, unit=\"m\", h=3) == \"6m2\" and area(2, h=3, unit=\"m\") == \"6m2\"\n    assert area.info() == {\"hits\": 2, \"misses\": 3, \"size\": 3} and len(calls) == 3\n    assert area.__name__ == \"area\" and area.__doc__ == \"Area of a rectangle.\"\n    area.clear()\n    assert area.info() == {\"hits\": 0, \"misses\": 0, \"size\": 0}\n    area(2, 3)\n    assert len(calls) == 4")

### Exercise: An LRU cache

Write `LRUCache(capacity)` using `collections.OrderedDict`: `get(key, default=None)` (a hit makes the key the most recently used), `put(key, value)` (adding or updating, also making it most recent, and evicting the **least** recently used key when there are more than `capacity` entries), `len()`, `keys()` (a list from least to most recently used), and counters `hits`, `misses` (counted by `get`) and `evictions`.

In [ ]:
from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lru():\n    \"least recently used\"\n    c = LRUCache(2)\n    c.put(\"a\", 1)\n    c.put(\"b\", 2)\n    assert c.get(\"a\") == 1 and c.keys() == [\"b\", \"a\"]\n    c.put(\"c\", 3)\n    assert c.keys() == [\"a\", \"c\"] and c.get(\"b\") is None and c.get(\"b\", 0) == 0\n    c.put(\"a\", 10)\n    c.put(\"d\", 4)\n    assert c.keys() == [\"a\", \"d\"] and c.get(\"a\") == 10 and len(c) == 2\n    assert (c.hits, c.misses, c.evictions) == (2, 2, 2)")

### Exercise: Results that expire

Write a decorator factory `ttl_cache(seconds, clock=time.monotonic)` for functions of hashable positional arguments: a cached result is returned while `clock()` is **less than** the time it was stored plus `seconds`; after that the function is called again and the new result stored. The wrapper has `purge()`, which removes expired entries and returns how many it removed, and `size()`.

In [ ]:
import functools
import time

def ttl_cache(seconds, clock=time.monotonic):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_ttl():\n    \"time to live\"\n    now = [100.0]\n    calls = []\n    @ttl_cache(10, clock=lambda: now[0])\n    def rate(currency):\n        calls.append(currency)\n        return {\"EUR\": 1.17, \"USD\": 1.27}[currency] + len(calls) / 100\n    first = rate(\"EUR\")\n    now[0] = 109.9\n    assert rate(\"EUR\") == first and calls == [\"EUR\"]\n    now[0] = 110.0\n    assert rate(\"EUR\") != first and calls == [\"EUR\", \"EUR\"]\n    rate(\"USD\")\n    now[0] = 121.0\n    assert rate.size() == 2 and rate.purge() == 2 and rate.size() == 0\n    import time\n    @ttl_cache(60)\n    def real(x):\n        return x * 2\n    assert real(2) == 4 and real(2) == 4")

### Exercise: Keys for any arguments

Write `freeze(value)`: a hashable version of nested data (lists and tuples to tuples, dicts to tuples of `(key, frozen value)` pairs sorted by key, sets to frozensets, other values unchanged), and a decorator `memo_any(func)` that caches calls even when the arguments are lists, dicts or sets, using frozen arguments (and sorted keyword arguments) as the key. Like `lru_cache`, it gives the wrapper a `calls` counter of how many times the real function ran.

In [ ]:
import functools

def freeze(value):
    ___

def memo_any(func):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_keys():\n    \"cache keys\"\n    assert freeze({\"b\": [1, 2], \"a\": {3}}) == ((\"a\", frozenset({3})), (\"b\", (1, 2)))\n    assert hash(freeze([{\"x\": [1]}, (2, [3])])) == hash((((\"x\", (1,)),), (2, (3,))))\n    @memo_any\n    def total(items, weights=None):\n        return sum(i * (weights or {}).get(i, 1) for i in items)\n    assert total([1, 2, 3]) == 6 and total([1, 2, 3]) == 6 and total.calls == 1\n    assert total([1, 2], weights={2: 10}) == 21 and total([1, 2], weights={2: 10}) == 21 and total.calls == 2\n    assert total([2, 1]) == 3 and total.calls == 3")

### Exercise: A cache that survives

Write a decorator factory `disk_cache(path)` for functions whose arguments and results can be stored as JSON. The key is `json.dumps([args, kwargs], sort_keys=True)`. On each miss, the result is stored in a dict which is written to the JSON file at `path`; when the decorator is applied, any existing file is loaded first, so a **new** decorated function (as in a new run of the program) reuses earlier results. The wrapper has `misses`, counting real calls in this run.

In [ ]:
import functools
import json
import os

def disk_cache(path):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_disk():\n    \"persistent caching\"\n    import os\n    if os.path.exists(\"cache.json\"):\n        os.remove(\"cache.json\")\n    def slow_square(n, label=\"sq\"):\n        return {\"label\": label, \"value\": n * n}\n    first = disk_cache(\"cache.json\")(slow_square)\n    assert first(4) == {\"label\": \"sq\", \"value\": 16} and first(4)[\"value\"] == 16 and first.misses == 1\n    first(5, label=\"x\")\n    second = disk_cache(\"cache.json\")(slow_square)\n    assert second(4)[\"value\"] == 16 and second(5, label=\"x\")[\"label\"] == \"x\" and second.misses == 0\n    second(6)\n    assert second.misses == 1\n    with open(\"cache.json\") as f:\n        assert len(json.load(f)) == 3\n    os.remove(\"cache.json\")")

### Bug hunt: A list as a cache key

`lru_cache` needs hashable arguments, so calling the cached function with a list crashes with `TypeError: unhashable type: 'list'`. Pass the items as a tuple so this prints `6 6 1`.

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def total(items):
    return sum(items)

basket = [1, 2, 3]
print(total(basket), total(basket), total.cache_info().hits)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"hashable keys\"\n    expect_output(\"6 6 1\")")

### Bug hunt: A stale answer

The cached function reads a global price list, which changes, but the cache keeps returning the old total: this prints `5 5`. Clear the cache when the prices change (with `cache_clear()`), so it prints `5 7`.

In [ ]:
from functools import cache

PRICES = {"tea": 2, "jam": 3}

@cache
def basket_total(*items):
    return sum(PRICES[i] for i in items)

before = basket_total("tea", "jam")
PRICES["tea"] = 4
after = basket_total("tea", "jam")
print(before, after)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"invalidation\"\n    expect_output(\"5 7\")")

### Bug hunt: A cached list that someone changed

The cache hands every caller the **same** list, and one caller appends to it, so the next caller gets a changed result: this prints `['tea', 'jam', 'extra']`. Make the cached function return a tuple (and the caller build its own list from it), so the cached value can't be changed and this prints `('tea', 'jam')`.

In [ ]:
from functools import cache

@cache
def default_basket():
    return ["tea", "jam"]

mine = default_basket()
mine.append("extra")
print(default_basket())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"immutable results\"\n    expect_output(\"('tea', 'jam')\")")

### Puzzle: Evicting the least recently used

Arrange the lines of an LRU `put` method using an `OrderedDict`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # True or False: it's safe to cache a function that returns the current time
q = ___   # the error lru_cache raises for a list argument, as a string
r = ___   # which entry an LRU cache evicts: "oldest added", "least recently used" or "largest"
s = ___   # hits after calling a cached f(1), f(2), f(1), f(1)
t = ___   # True or False: lru_cache on a method can keep instances alive

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is False and q == \"TypeError\" and r == \"least recently used\", \"p, q and r\"\n    assert s == 2 and t is True, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: caching with dependencies

Build a small reactive store whose cached computations are invalidated **only** when data they actually read changes, like cells in a spreadsheet.

- `Store(**values)`: `get(name)` and `set(name, value)` for plain values.
- `@store.computed` turns a function of no arguments into a **computed value**, registered under the function's name and readable with `store.get(name)` too. The first read runs the function and caches the result; while it runs, the store records every name it reads with `get` (plain values **and** other computed values) as its dependencies.
- `set(name, value)` invalidates every computed value that depends on `name`, **directly or through other computed values**, and nothing else. Setting a value to one equal to the current value invalidates nothing.
- `runs(name)`: how many times that computed value's function has run. `dependencies(name)`: the sorted names it read on its last run.

Computed values that read different names on different runs (because of an `if`) must have their dependencies replaced on each run.

In [ ]:
class Store:
    def __init__(self, **values):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_reactive():\n    \"dependency tracking\"\n    s = Store(price=10, qty=2, tax=0.2, unused=0)\n    @s.computed\n    def subtotal():\n        return s.get(\"price\") * s.get(\"qty\")\n    @s.computed\n    def total():\n        return round(s.get(\"subtotal\") * (1 + s.get(\"tax\")), 2)\n    assert s.get(\"total\") == 24.0 and s.get(\"total\") == 24.0 and s.runs(\"total\") == 1 and s.runs(\"subtotal\") == 1\n    assert s.dependencies(\"total\") == [\"subtotal\", \"tax\"] and s.dependencies(\"subtotal\") == [\"price\", \"qty\"]\n    s.set(\"unused\", 5)\n    s.get(\"total\")\n    assert s.runs(\"total\") == 1\n    s.set(\"tax\", 0.5)\n    assert s.get(\"total\") == 30.0 and s.runs(\"total\") == 2 and s.runs(\"subtotal\") == 1\n    s.set(\"qty\", 3)\n    assert s.get(\"total\") == 45.0 and s.runs(\"subtotal\") == 2 and s.runs(\"total\") == 3\n    s.set(\"qty\", 3)\n    s.get(\"total\")\n    assert s.runs(\"total\") == 3\n\ndef test_changing_dependencies():\n    \"dependencies follow the last run\"\n    s = Store(use_discount=False, price=100, discount=10)\n    @s.computed\n    def final():\n        if s.get(\"use_discount\"):\n            return s.get(\"price\") - s.get(\"discount\")\n        return s.get(\"price\")\n    assert s.get(\"final\") == 100 and s.dependencies(\"final\") == [\"price\", \"use_discount\"]\n    s.set(\"discount\", 50)\n    s.get(\"final\")\n    assert s.runs(\"final\") == 1\n    s.set(\"use_discount\", True)\n    assert s.get(\"final\") == 50 and s.dependencies(\"final\") == [\"discount\", \"price\", \"use_discount\"]\n    s.set(\"discount\", 20)\n    assert s.get(\"final\") == 80 and s.runs(\"final\") == 3")

## Warm-up

The second `"tea"` was a hit. But with room for only two entries, adding `"milk"` evicted the least recently used entry, `"jam"`, so asking for `"jam"` again meant computing it again. A cache is a trade: memory for time, and how much memory you allow changes how much time you save.

## When caching is safe

Caching a function returns the **old answer** instead of running the function. That's only correct when the function is **pure** enough:

- the result depends **only on the arguments** (not on the time, a file, a database, global variables or random numbers), or you accept a slightly stale result;
- calling it has **no side effects** that you need to happen every time (printing, writing, sending, counting);
- the result isn't **mutated** by callers: everyone who asks gets the **same object**, so if one caller appends to a cached list, every later caller sees the change (Week 62). Return immutable results, or copies.

Caching pays when the function is **expensive** relative to a lookup, and the **same arguments come back** often (a high **hit rate**). Typical wins: parsing, recursive computations with overlapping subproblems (Week 52's dynamic programming), conversions, lookups over a slow network or disk, and results of database queries that change rarely.

## Memoising and its statistics

A memoising decorator is a dict from arguments to results wrapped around a function. Making it useful in practice means a few additions, which `functools.lru_cache` has and your own should too:

- **statistics**: hits, misses and the current size, so you can check the cache is actually working (a hit rate near zero means the cache only costs memory);
- **`cache_clear()`**, to empty it (for tests, and when data changes);
- **`functools.wraps`**, so the wrapped function keeps its name and docstring;
- handling **keyword arguments**: `f(1, b=2)` and `f(1, 2)` are separate keys in `lru_cache`, which is safe but can waste entries.

`@functools.cache` (unbounded) and `@functools.lru_cache(maxsize=128)` are implemented in C and are the right choice when your arguments are hashable; write your own when you need different keys, expiry, or storage.

## Bounded caches: LRU and TTL

An unbounded cache is a memory leak waiting to happen (Week 62): every new argument adds an entry for ever. Two common bounds:

- **LRU (least recently used)**: keep at most `n` entries; when full, evict the entry that was used longest ago. Recently used items tend to be used again, so this keeps the useful ones. `collections.OrderedDict` makes it simple: `move_to_end(key)` on each use and `popitem(last=False)` to evict the oldest. (Week 44 built one from a hash map and a linked list.)
- **TTL (time to live)**: each entry is valid for a fixed time after it was stored, then treated as missing. That bounds **staleness** for data that changes, such as exchange rates or a list of online users. Store `(expiry_time, value)` and compare with the clock on each lookup; pass the clock in, so tests can control time.

Real caches often combine both: at most `n` entries, each for at most `t` seconds.

## Cache keys

The key must identify the question **exactly**: two calls get the same key if and only if they must have the same answer.

- Arguments must be **hashable** to be dict keys: `lru_cache` raises `TypeError: unhashable type: 'list'` for list arguments. Either pass tuples, or build a key by **freezing** values: lists to tuples, dicts to sorted tuples of items, sets to frozensets (Week 62's `freeze`).
- **Keyword order** shouldn't matter: sort keyword arguments into the key.
- Include **everything the result depends on**, such as a version number of the data or the user's language, or two different questions will share an answer.
- Don't include things that **don't** matter (a request id, a timestamp), or the hit rate drops to zero.
- For a cache stored on disk or shared between processes, the key must be a **string** or bytes: `json.dumps(key, sort_keys=True)`, or a hash of it (`hashlib.sha256(text.encode()).hexdigest()`).

## Invalidation

"There are only two hard things in Computer Science: cache invalidation and naming things" (a saying credited to Phil Karlton). A cached result becomes **wrong** when the data it was computed from changes. Ways to cope, simplest first:

- **Don't cache** data that changes, or cache it only within one request or one run.
- **Expire** entries after a time (TTL) and accept bounded staleness.
- **Clear** the whole cache when anything changes (`cache_clear()`): simple, and fine if changes are rare.
- **Version the data**: include a version number in every key, and bump it on change; old entries are simply never asked for again (and an LRU bound removes them eventually).
- **Track dependencies**: record which pieces of data each cached result read, and remove only the entries that depend on what changed (today's challenge). Spreadsheets, build tools and user-interface frameworks work this way.

One more trap: caching **methods** with `@lru_cache` puts `self` in every key, so the cache keeps every instance alive for ever. Use `functools.cached_property` (stored on the instance, Week 63) or a per-instance dict instead.

## Practice

## Quiz

## Daily challenge

## Before you go

Find a function in your own code that's called repeatedly with the same arguments. Is it pure? Would `@functools.cache` be safe, and what would its hit rate be? Tomorrow: vectorisation with NumPy, which moves whole loops into fast compiled code.